# V9 — Partie 2 : Post-traitement, Data Quality et Validation

Cette partie **ne charge pas Qwen, Transformers ni CUDA**. Elle lit uniquement les JSON `DOM_EXTRACTION_V1` produits par la Partie 1.

Principes :
- `raw_data` reste immuable ;
- chaque transformation est tracée (`raw`, `normalized`, `rule`, `status`) ;
- les règles Python peuvent évoluer et être rejouées sans relire les PDF ;
- les divergences inter-documents génèrent des anomalies / demandes de relecture, mais **n'appellent jamais Qwen automatiquement** ;
- le résultat principal du test est un classeur de validation sur 10 dossiers.


## 1. Imports et configuration


In [ ]:
import hashlib
import json
import math
import re
from collections import defaultdict
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

SCHEMA_VERSION = 'DOM_EXTRACTION_V1'
EXPECTED_FIELD_SCHEMA_HASH = 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'
POSTPROCESS_VERSION = 'DOM_V9_0_PART2_POSTPROCESS_V1'
NORMALIZATION_VERSION = 'NORM_V1_0'
MAX_DOSSIERS = 10   # None après validation

OUTPUT_ROOT = Path('/mnt/data/domiciliations_v9')
RAW_JSON_DIR = OUTPUT_ROOT / '01_extraction_raw' / 'json_dossiers'
POST_ROOT = OUTPUT_ROOT / '02_postprocessing'
PROCESSED_JSON_DIR = POST_ROOT / 'processed_json'
VALIDATION_XLSX = POST_ROOT / 'validation_domiciliations_v9.xlsx'
DOSSIERS_CSV = POST_ROOT / 'dossiers_a_valider.csv'
FIELDS_CSV = POST_ROOT / 'champs_detail.csv'
ANOMALIES_CSV = POST_ROOT / 'anomalies.csv'
RETRY_CSV = POST_ROOT / 'vlm_retry_requests.csv'

POST_ROOT.mkdir(parents=True,exist_ok=True)
PROCESSED_JSON_DIR.mkdir(parents=True,exist_ok=True)

print('RAW :',RAW_JSON_DIR)
print('Post:',POST_ROOT)


## 2. Schéma — mêmes 99 champs V8.1 / V9


In [ ]:
FIELD_SCHEMA = {'ENGAGEMENT_DOMICILIATION': ['DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL', 'DOM_ADRESSE_CLIENT', 'DOM_AGENCE_DOMICILIATAIRE', 'DOM_NUMERO_CONTRAT', 'DOM_DUREE_CONTRAT_MOIS', 'DOM_DATE_DEBUT_CONTRAT', 'DOM_DATE_FIN_CONTRAT', 'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', 'DOM_ADRESSE_EMPLOYEUR', 'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE', 'DOM_TAUX_TRANSFERABLE', 'DOM_MONTANT_TOTAL_DOMICILIE', 'DOM_DATE_SIGNATURE'], 'CONTRAT_TRAVAIL': ['CTR_REFERENCE_DOCUMENT', 'CTR_TYPE', 'CTR_EMPLOYEUR', 'CTR_ACTIVITE_EMPLOYEUR', 'CTR_DUREE_MOIS', 'CTR_DATE_DEBUT_CONTRAT', 'CTR_POSTE', 'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_PERE_NOM_PRENOM', 'CTR_MERE_NOM_PRENOM', 'CTR_NATIONALITE', 'CTR_DATE_NAISSANCE', 'CTR_LIEU_PAYS_NAISSANCE', 'CTR_ADRESSE_ALGERIE', 'CTR_QUALIFICATION', 'CTR_NUMERO_PERMIS_TRAVAIL', 'CTR_DATE_DELIVRANCE_PERMIS', 'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS', 'CTR_SALAIRE_BRUT', 'CTR_SALAIRE_NET', 'CTR_AFFILIATION_SS', 'CTR_NUMERO_EMPLOYEUR', 'CTR_DATE_SIGNATURE', 'CTR_REFERENCE_DOMICILIATION', 'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTR_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTR_CACHET_EMPLOYEUR_PRESENT'], 'CONTRAT_SPECIFIQUE': ['CTS_REFERENCE_DOCUMENT', 'CTS_SAP_ID', 'CTS_EMPLOYEUR', 'CTS_ACTIVITE_EMPLOYEUR', 'CTS_DUREE_MOIS', 'CTS_DATE_DEBUT_CONTRAT', 'CTS_POSTE', 'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_PERE_NOM_PRENOM', 'CTS_MERE_NOM_PRENOM', 'CTS_NATIONALITE', 'CTS_DATE_NAISSANCE', 'CTS_LIEU_PAYS_NAISSANCE', 'CTS_ADRESSE_ALGERIE', 'CTS_QUALIFICATION', 'CTS_NUMERO_PERMIS_TRAVAIL', 'CTS_DATE_DELIVRANCE_PERMIS', 'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS', 'CTS_LIGNE_SALAIRE_BRUTE', 'CTS_SALAIRE_NET', 'CTS_SALAIRE_NET_ANCIEN', 'CTS_MENTION_AU_LIEU_DE_PRESENTE', 'CTS_PART_TRANSFERABLE', 'CTS_PART_PAYABLE_DZD', 'CTS_NUMERO_SS_PAYS_ORIGINE', 'CTS_NUMERO_SS_ALGERIE', 'CTS_DATE_DOCUMENT', 'CTS_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTS_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTS_CACHET_EMPLOYEUR_PRESENT', 'CTS_VISA_INSPECTION_TRAVAIL_PRESENT'], 'TITRE_TRAVAIL': ['TTR_NUMERO_PERMIS', 'TTR_NUMERO_MANUSCRIT', 'TTR_POSTE', 'TTR_DUREE', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'TTR_LIEU_TRAVAIL', 'TTR_EMPLOYEUR', 'TTR_ADRESSE_EMPLOYEUR', 'TTR_FAIT_A', 'TTR_DATE_DELIVRANCE', 'TTR_NOM', 'TTR_PRENOM', 'TTR_DATE_NAISSANCE', 'TTR_LIEU_NAISSANCE', 'TTR_PAYS', 'TTR_NATIONALITE', 'TTR_QUALIFICATION', 'TTR_DATE_ENTREE_ALGERIE', 'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT'], 'PERMIS_TRAVAIL_COUVERTURE': ['PTR_NUMERO_SERIE', 'PTR_WILAYA', 'PTR_CACHET_DIRECTION_EMPLOI_PRESENT']}

ALL_FIELDS=[f for fields in FIELD_SCHEMA.values() for f in fields]
assert len(ALL_FIELDS)==99 and len(set(ALL_FIELDS))==99
schema_hash=hashlib.sha256(json.dumps(FIELD_SCHEMA,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
assert schema_hash==EXPECTED_FIELD_SCHEMA_HASH
print('✅ Schéma 99 champs vérifié |',schema_hash[:16]+'…')


## 3. Typage des champs — configuration évolutive


In [ ]:
AMOUNT_FIELDS={
    'DOM_SALAIRE_NET_MENSUEL','DOM_PART_TRANSFERABLE','DOM_MONTANT_TOTAL_DOMICILIE',
    'CTR_SALAIRE_BRUT','CTR_SALAIRE_NET','CTS_SALAIRE_NET','CTS_SALAIRE_NET_ANCIEN',
    'CTS_PART_TRANSFERABLE','CTS_PART_PAYABLE_DZD',
}
PERCENT_FIELDS={'DOM_TAUX_TRANSFERABLE'}
INTEGER_FIELDS={'DOM_DUREE_CONTRAT_MOIS','CTR_DUREE_MOIS','CTS_DUREE_MOIS'}
BOOLEAN_FIELDS={
    'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE','CTR_SIGNATURE_EMPLOYEUR_PRESENTE','CTR_CACHET_EMPLOYEUR_PRESENT',
    'CTS_MENTION_AU_LIEU_DE_PRESENTE','CTS_SIGNATURE_TRAVAILLEUR_PRESENTE','CTS_SIGNATURE_EMPLOYEUR_PRESENTE',
    'CTS_CACHET_EMPLOYEUR_PRESENT','CTS_VISA_INSPECTION_TRAVAIL_PRESENT',
    'TTR_PHOTO_PRESENTE','TTR_CACHET_PRESENT','PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
}
DATE_FIELDS={f for f in ALL_FIELDS if '_DATE_' in f or f.startswith('TTR_DATE_') or f in {'DOM_DATE_SIGNATURE','CTS_DATE_DOCUMENT'}}
REFERENCE_FIELDS={
    'DOM_COMPTE_LOCAL','DOM_NUMERO_CONTRAT','CTR_REFERENCE_DOCUMENT','CTR_NUMERO_PERMIS_TRAVAIL',
    'CTR_REFERENCE_DOMICILIATION','CTS_REFERENCE_DOCUMENT','CTS_SAP_ID','CTS_NUMERO_PERMIS_TRAVAIL',
    'TTR_NUMERO_PERMIS','TTR_NUMERO_MANUSCRIT','PTR_NUMERO_SERIE',
}

FIELD_TYPES={}
for f in ALL_FIELDS:
    if f in AMOUNT_FIELDS: FIELD_TYPES[f]='amount'
    elif f in PERCENT_FIELDS: FIELD_TYPES[f]='percentage'
    elif f in INTEGER_FIELDS: FIELD_TYPES[f]='integer'
    elif f in BOOLEAN_FIELDS: FIELD_TYPES[f]='boolean'
    elif f in DATE_FIELDS: FIELD_TYPES[f]='date'
    elif f in REFERENCE_FIELDS: FIELD_TYPES[f]='reference'
    else: FIELD_TYPES[f]='text'

print(pd.Series(FIELD_TYPES).value_counts())


## 4. Normalisation traçable


In [ ]:
NULL_TEXTS={'','NULL','NONE','N/A','NA','NÉANT','NEANT','ILLISIBLE','NON LISIBLE'}

def result(raw,normalized,status,rule,message=None):
    return {'raw':raw,'normalized':normalized,'status':status,'rule':rule,
            'changed':(normalized != raw),'message':message}


def normalize_amount_trace(raw):
    if raw is None: return result(raw,None,'MISSING','AMOUNT_NULL')
    if isinstance(raw,(int,float,Decimal)) and not isinstance(raw,bool):
        try: return result(raw,round(float(raw),2),'RAW_OK','AMOUNT_NUMERIC')
        except Exception: pass
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','AMOUNT_NULL_TEXT')

    # Retirer uniquement des éléments non numériques usuels de devise, sans modifier les chiffres.
    t=s.upper().replace('DZD','').replace('DA','').replace('EUR','').replace('€','').strip()
    t=t.replace("'",'').replace('’','').replace(' ','')
    neg=t.startswith('-'); t=t.lstrip('+-')
    if not re.fullmatch(r'[0-9.,]+',t or ''):
        return result(raw,None,'REVIEW','AMOUNT_NON_NUMERIC','caractère non numérique ambigu')

    dots=t.count('.'); commas=t.count(',')
    dec_sep=None; rule=''
    if dots and commas:
        last_dot=t.rfind('.'); last_comma=t.rfind(',')
        candidate='.' if last_dot>last_comma else ','
        tail=t.split(candidate)[-1]
        if len(tail) in (1,2): dec_sep=candidate; rule='AMOUNT_MIXED_LAST_DECIMAL'
        else: return result(raw,None,'REVIEW','AMOUNT_MIXED_AMBIGUOUS')
    elif dots>1 or commas>1:
        sep='.' if dots else ','; groups=t.split(sep); tail=groups[-1]
        if len(tail) in (1,2): dec_sep=sep; rule='AMOUNT_MULTI_SEPARATOR_LAST_DECIMAL'
        elif all(len(g)==3 for g in groups[1:]): dec_sep=None; rule='AMOUNT_MULTI_THOUSANDS'
        else: return result(raw,None,'REVIEW','AMOUNT_MULTI_AMBIGUOUS')
    elif dots==1 or commas==1:
        sep='.' if dots else ','; left,right=t.split(sep)
        if len(right) in (1,2): dec_sep=sep; rule='AMOUNT_SINGLE_DECIMAL'
        elif len(right)==3:
            # 23,340 peut signifier 23 340 ou 23.340 : ne pas inventer.
            return result(raw,None,'REVIEW','AMOUNT_SINGLE_3DIGITS_AMBIGUOUS')
        else:
            return result(raw,None,'REVIEW','AMOUNT_SINGLE_AMBIGUOUS')
    else:
        rule='AMOUNT_INTEGER'

    if dec_sep:
        pos=t.rfind(dec_sep)
        int_part=re.sub(r'[.,]','',t[:pos]); dec=t[pos+1:]
        canonical=int_part+'.'+dec
    else:
        canonical=re.sub(r'[.,]','',t)
    if neg: canonical='-'+canonical
    try:
        val=round(float(Decimal(canonical)),2)
    except (InvalidOperation,ValueError):
        return result(raw,None,'REVIEW','AMOUNT_PARSE_FAILED')
    return result(raw,val,'AUTO_OK' if str(raw)!=str(val) else 'RAW_OK',rule)


DATE_FORMATS=['%d/%m/%Y','%d-%m-%Y','%d.%m.%Y','%Y-%m-%d','%Y/%m/%d','%Y.%m.%d']
def normalize_date_trace(raw):
    if raw is None: return result(raw,None,'MISSING','DATE_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','DATE_NULL_TEXT')
    s=re.sub(r'\s+','',s)
    for fmt in DATE_FORMATS:
        try:
            dt=datetime.strptime(s,fmt)
            out=dt.strftime('%d/%m/%Y')
            return result(raw,out,'RAW_OK' if s==out else 'AUTO_OK','DATE_'+fmt.replace('%',''))
        except ValueError:
            pass
    return result(raw,None,'REVIEW','DATE_UNPARSEABLE')


def normalize_integer_trace(raw):
    if raw is None: return result(raw,None,'MISSING','INTEGER_NULL')
    if isinstance(raw,int) and not isinstance(raw,bool): return result(raw,raw,'RAW_OK','INTEGER_NUMERIC')
    s=str(raw).strip(); m=re.fullmatch(r'\s*(\d+)\s*(?:mois)?\s*',s,flags=re.I)
    if not m: return result(raw,None,'REVIEW','INTEGER_AMBIGUOUS')
    v=int(m.group(1)); return result(raw,v,'RAW_OK' if str(v)==s else 'AUTO_OK','INTEGER_EXTRACT')


def normalize_percentage_trace(raw):
    if raw is None: return result(raw,None,'MISSING','PERCENT_NULL')
    s=str(raw).strip().replace('\u00a0',' ')
    has_pct='%' in s
    s=s.replace('%','').replace(' ','').replace(',','.')
    if not re.fullmatch(r'[+-]?\d+(?:\.\d+)?',s): return result(raw,None,'REVIEW','PERCENT_AMBIGUOUS')
    v=float(s)
    if not has_pct and 0 < v <= 1:
        v*=100; rule='PERCENT_FRACTION_TO_PERCENT'
    else: rule='PERCENT_DIRECT'
    if not (0 <= v <= 100): return result(raw,None,'REVIEW','PERCENT_OUT_OF_RANGE')
    v=round(v,2); return result(raw,v,'AUTO_OK' if str(raw).strip()!=str(v) else 'RAW_OK',rule)


def normalize_boolean_trace(raw):
    if raw is None: return result(raw,None,'MISSING','BOOL_NULL')
    if isinstance(raw,bool): return result(raw,raw,'RAW_OK','BOOL_NATIVE')
    s=str(raw).strip().upper()
    if s in {'TRUE','VRAI','OUI','YES','1'}: return result(raw,True,'AUTO_OK','BOOL_TRUE_TEXT')
    if s in {'FALSE','FAUX','NON','NO','0'}: return result(raw,False,'AUTO_OK','BOOL_FALSE_TEXT')
    return result(raw,None,'REVIEW','BOOL_AMBIGUOUS')


def normalize_reference_trace(raw):
    if raw is None: return result(raw,None,'MISSING','REF_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','REF_NULL_TEXT')
    # Conservateur : espaces périphériques et autour de / - uniquement.
    out=re.sub(r'\s*([/\-])\s*',r'\1',re.sub(r'\s+',' ',s)).strip()
    return result(raw,out,'AUTO_OK' if out!=s else 'RAW_OK','REF_SPACING_ONLY')


def normalize_text_trace(raw):
    if raw is None: return result(raw,None,'MISSING','TEXT_NULL')
    s=str(raw).replace('\u00a0',' ').strip()
    if s.upper() in NULL_TEXTS: return result(raw,None,'MISSING','TEXT_NULL_TEXT')
    # Pas de correction orthographique / casse.
    return result(raw,s,'AUTO_OK' if s!=raw else 'RAW_OK','TEXT_TRIM_ONLY')


def normalize_field_trace(field,raw):
    typ=FIELD_TYPES.get(field,'text')
    return {
        'amount':normalize_amount_trace,
        'date':normalize_date_trace,
        'integer':normalize_integer_trace,
        'percentage':normalize_percentage_trace,
        'boolean':normalize_boolean_trace,
        'reference':normalize_reference_trace,
        'text':normalize_text_trace,
    }[typ](raw)

# Cas demandé explicitement : aucune relance Qwen nécessaire.
_t=normalize_amount_trace('23.340.43')
assert _t['normalized']==23340.43 and _t['status']=='AUTO_OK', _t
assert normalize_amount_trace('23,340.43')['normalized']==23340.43
assert normalize_amount_trace('23.340,43')['normalized']==23340.43
assert normalize_amount_trace('23 340,43')['normalized']==23340.43
assert normalize_amount_trace('23,340')['status']=='REVIEW'
print('✅ Normalisation V1 testée : 23.340.43 -> 23340.43')


## 5. Contrôles croisés — sans appel Qwen


In [ ]:
CROSS_DOCUMENT_GROUPS=[
    {'name':'SALAIRE_NET','kind':'amount','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_SALAIRE_NET_MENSUEL','CONTRAT_TRAVAIL':'CTR_SALAIRE_NET','CONTRAT_SPECIFIQUE':'CTS_SALAIRE_NET'}},
    {'name':'PART_TRANSFERABLE','kind':'amount','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_PART_TRANSFERABLE','CONTRAT_SPECIFIQUE':'CTS_PART_TRANSFERABLE'}},
    {'name':'DATE_DEBUT_CONTRAT','kind':'date','fields':{
        'ENGAGEMENT_DOMICILIATION':'DOM_DATE_DEBUT_CONTRAT','CONTRAT_TRAVAIL':'CTR_DATE_DEBUT_CONTRAT','CONTRAT_SPECIFIQUE':'CTS_DATE_DEBUT_CONTRAT'}},
    {'name':'NUMERO_PERMIS','kind':'reference','fields':{
        'CONTRAT_TRAVAIL':'CTR_NUMERO_PERMIS_TRAVAIL','CONTRAT_SPECIFIQUE':'CTS_NUMERO_PERMIS_TRAVAIL','TITRE_TRAVAIL':'TTR_NUMERO_PERMIS'}},
    {'name':'DATE_DEBUT_PERMIS','kind':'date','fields':{
        'CONTRAT_TRAVAIL':'CTR_DATE_DEBUT_VALIDITE_PERMIS','CONTRAT_SPECIFIQUE':'CTS_DATE_DEBUT_VALIDITE_PERMIS','TITRE_TRAVAIL':'TTR_DATE_DEBUT'}},
    {'name':'DATE_FIN_PERMIS','kind':'date','fields':{
        'CONTRAT_TRAVAIL':'CTR_DATE_FIN_VALIDITE_PERMIS','CONTRAT_SPECIFIQUE':'CTS_DATE_FIN_VALIDITE_PERMIS','TITRE_TRAVAIL':'TTR_DATE_FIN'}},
]

def comparable(v): return v not in (None,'')
def same_value(kind,a,b):
    if not comparable(a) or not comparable(b): return True
    if kind=='amount': return abs(float(a)-float(b))<=0.01
    return str(a)==str(b)


## 6. Traitement d’un JSON RAW


In [ ]:
def validate_raw_contract(d):
    return (d.get('schema_version')==SCHEMA_VERSION and
            d.get('field_schema_hash')==EXPECTED_FIELD_SCHEMA_HASH and
            isinstance(d.get('page_records'),list))


def process_raw_dossier(d):
    if not validate_raw_contract(d):
        raise ValueError(f"Contrat RAW incompatible : {d.get('source_file')}")
    source=d['source_file']
    processed_records=[]; field_rows=[]; anomalies=[]; retry=[]

    for rec in d['page_records']:
        dt=rec.get('doc_type'); page=rec.get('page_num'); raw=dict(rec.get('raw_data') or {})
        normalized={}; trace={}
        for field in FIELD_SCHEMA.get(dt,[]):
            tr=normalize_field_trace(field,raw.get(field))
            trace[field]=tr; normalized[field]=tr['normalized']
            field_rows.append({
                'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,'CHAMP':field,'TYPE_CHAMP':FIELD_TYPES[field],
                'RAW':tr['raw'],'NORMALIZED':tr['normalized'],'STATUS':tr['status'],'RULE':tr['rule'],
                'CHANGED':tr['changed'],'MESSAGE':tr.get('message'),
            })
            if tr['status']=='REVIEW':
                anomalies.append({'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,'TYPE_ANOMALIE':'FORMAT_REVIEW',
                                  'CHAMP':field,'VALEUR_RAW':tr['raw'],'VALEUR_NORMALISEE':tr['normalized'],
                                  'MOTIF':tr['rule']})
                retry.append({'FICHIER':source,'PAGE':page,'TYPE_DOCUMENT':dt,'CHAMP':field,
                              'VALEUR_RAW':tr['raw'],'MOTIF':'FORMAT_AMBIGU','RULE':tr['rule']})
        p=dict(rec); p['normalized_data']=normalized; p['normalization_trace']=trace
        processed_records.append(p)

    # cross-check uniquement sur valeurs normalisées valides
    by_type=defaultdict(list)
    for r in processed_records: by_type[r.get('doc_type')].append(r)
    for group in CROSS_DOCUMENT_GROUPS:
        vals=[]
        for dt,field in group['fields'].items():
            for r in by_type.get(dt,[]):
                v=(r.get('normalized_data') or {}).get(field)
                if comparable(v): vals.append((dt,field,r.get('page_num'),v,(r.get('raw_data') or {}).get(field)))
        if len(vals)>=2:
            base=vals[0][3]
            if any(not same_value(group['kind'],base,x[3]) for x in vals[1:]):
                detail=' | '.join(f'{dt}.{field}@p{p}={v}' for dt,field,p,v,raw in vals)
                anomalies.append({'FICHIER':source,'PAGE':None,'TYPE_DOCUMENT':'MULTI','TYPE_ANOMALIE':'CROSS_DOCUMENT_CONFLICT',
                                  'CHAMP':group['name'],'VALEUR_RAW':None,'VALEUR_NORMALISEE':detail,
                                  'MOTIF':'valeurs divergentes entre documents'})
                for dt,field,p,v,raw in vals:
                    retry.append({'FICHIER':source,'PAGE':p,'TYPE_DOCUMENT':dt,'CHAMP':field,
                                  'VALEUR_RAW':raw,'MOTIF':'CROSS_DOCUMENT_CONFLICT','RULE':group['name']})

    out={
        'schema_version':SCHEMA_VERSION,'source_file':source,'source_sha256':d.get('source_sha256'),
        'source_pipeline_version':d.get('pipeline_version'),'postprocess_version':POSTPROCESS_VERSION,
        'normalization_version':NORMALIZATION_VERSION,'processed_at':datetime.now().isoformat(timespec='seconds'),
        'page_records':processed_records,'anomalies':anomalies,'retry_requests':retry,
        'source_stats':d.get('stats') or {},
    }
    return out,field_rows,anomalies,retry


def first_non_null(*values):
    for v in values:
        if v not in (None,''): return v
    return None


def consolidate_for_validation(processed):
    row={'FICHIER':processed['source_file']}
    records=processed['page_records']
    row['NB_PAGES']=len(set(r.get('page_num') for r in records if r.get('page_num') is not None))
    row['TYPES_DOCUMENTS']=' | '.join(dict.fromkeys(str(r.get('doc_type')) for r in records))
    sources={}
    for r in records:
        dt=r.get('doc_type'); page=r.get('page_num'); nd=r.get('normalized_data') or {}
        for f,v in nd.items():
            if f not in row or row.get(f) in (None,''):
                row[f]=v; sources[f]=f'{dt}@p{page}'
    for f in ALL_FIELDS: row.setdefault(f,None)

    # Champs de synthèse conservateurs pour la validation humaine.
    row['NOM_TRAVAILLEUR_REFERENCE']=first_non_null(
        row.get('CTR_NOM_PRENOM_TRAVAILLEUR'),row.get('CTS_NOM_PRENOM_TRAVAILLEUR'),
        ' '.join(x for x in [str(row.get('TTR_NOM') or '').strip(),str(row.get('TTR_PRENOM') or '').strip()] if x) or None)
    row['NUMERO_PERMIS_REFERENCE']=first_non_null(row.get('TTR_NUMERO_PERMIS'),row.get('CTR_NUMERO_PERMIS_TRAVAIL'),
                                                  row.get('CTS_NUMERO_PERMIS_TRAVAIL'),row.get('PTR_NUMERO_SERIE'))
    row['DATE_DEBUT_CONTRAT_REFERENCE']=first_non_null(row.get('DOM_DATE_DEBUT_CONTRAT'),row.get('CTR_DATE_DEBUT_CONTRAT'),row.get('CTS_DATE_DEBUT_CONTRAT'))
    row['DATE_FIN_CONTRAT_REFERENCE']=first_non_null(row.get('DOM_DATE_FIN_CONTRAT'))
    row['SALAIRE_REFERENCE']=first_non_null(row.get('CTS_SALAIRE_NET'),row.get('CTR_SALAIRE_NET'),row.get('DOM_SALAIRE_NET_MENSUEL'))
    row['PART_TRANSFERABLE_REFERENCE']=first_non_null(row.get('DOM_PART_TRANSFERABLE'),row.get('CTS_PART_TRANSFERABLE'))
    row['NB_ANOMALIES']=len(processed.get('anomalies') or [])
    row['NB_RETRY_REQUESTS']=len(processed.get('retry_requests') or [])
    row['STATUT_VALIDATION']='A_VALIDER'
    row['COMMENTAIRE_VALIDATION']=None
    return row,sources


## 7. Exécution Partie 2 et classeur de validation


In [ ]:
json_files=sorted(RAW_JSON_DIR.glob('*.json'))
if MAX_DOSSIERS is not None:
    json_files=json_files[:int(MAX_DOSSIERS)]
print('JSON RAW sélectionnés :',len(json_files))

all_dossiers=[]; all_fields=[]; all_anomalies=[]; all_retry=[]; all_docs=[]; errors=[]
for i,p in enumerate(json_files,1):
    try:
        raw=json.loads(p.read_text(encoding='utf-8'))
        processed,field_rows,anomalies,retry=process_raw_dossier(raw)
        row,sources=consolidate_for_validation(processed)
        all_dossiers.append(row); all_fields.extend(field_rows); all_anomalies.extend(anomalies); all_retry.extend(retry)
        for r in processed['page_records']:
            all_docs.append({'FICHIER':processed['source_file'],'PAGE':r.get('page_num'),'TYPE_DOCUMENT':r.get('doc_type'),
                             'STATUT_EXTRACTION':r.get('extraction_status'),'TAUX_REMPLISSAGE':r.get('extraction_taux_remplissage'),
                             'CRITICAL_FIELDS_MISSING':' | '.join(r.get('critical_fields_missing') or []),
                             'STRATEGIES':' > '.join(s.get('nom','') for s in (r.get('extraction_strategies') or [])),
                             'TOKENS_IN':r.get('extraction_tokens_in'),'TOKENS_OUT':r.get('extraction_tokens_out')})
        outpath=PROCESSED_JSON_DIR/p.name
        outpath.write_text(json.dumps(processed,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
        print(f'[{i}/{len(json_files)}] ✅ {p.name} | anomalies={len(anomalies)}')
    except Exception as exc:
        errors.append({'FICHIER':p.name,'ERREUR':repr(exc)})
        print(f'[{i}/{len(json_files)}] ❌ {p.name}: {exc}')

DF_DOSSIERS=pd.DataFrame(all_dossiers)
DF_FIELDS=pd.DataFrame(all_fields)
DF_ANOMALIES=pd.DataFrame(all_anomalies)
DF_RETRY=pd.DataFrame(all_retry).drop_duplicates() if all_retry else pd.DataFrame(columns=['FICHIER','PAGE','TYPE_DOCUMENT','CHAMP','VALEUR_RAW','MOTIF','RULE'])
DF_DOCS=pd.DataFrame(all_docs)
DF_ERRORS=pd.DataFrame(errors)

DF_DOSSIERS.to_csv(DOSSIERS_CSV,index=False,encoding='utf-8-sig')
DF_FIELDS.to_csv(FIELDS_CSV,index=False,encoding='utf-8-sig')
DF_ANOMALIES.to_csv(ANOMALIES_CSV,index=False,encoding='utf-8-sig')
DF_RETRY.to_csv(RETRY_CSV,index=False,encoding='utf-8-sig')

with pd.ExcelWriter(VALIDATION_XLSX,engine='openpyxl') as writer:
    DF_DOSSIERS.to_excel(writer,sheet_name='DOSSIERS_A_VALIDER',index=False)
    DF_FIELDS.to_excel(writer,sheet_name='CHAMPS_DETAIL',index=False)
    DF_ANOMALIES.to_excel(writer,sheet_name='ANOMALIES',index=False)
    DF_RETRY.to_excel(writer,sheet_name='VLM_RETRY_REQUESTS',index=False)
    DF_DOCS.to_excel(writer,sheet_name='DOCUMENTS',index=False)
    DF_ERRORS.to_excel(writer,sheet_name='ERREURS',index=False)
    pd.DataFrame([
        {'PARAMETRE':'schema_version','VALEUR':SCHEMA_VERSION},
        {'PARAMETRE':'field_schema_hash','VALEUR':EXPECTED_FIELD_SCHEMA_HASH},
        {'PARAMETRE':'postprocess_version','VALEUR':POSTPROCESS_VERSION},
        {'PARAMETRE':'normalization_version','VALEUR':NORMALIZATION_VERSION},
    ]).to_excel(writer,sheet_name='PARAMETRES',index=False)

print('\n✅ Classeur validation :',VALIDATION_XLSX)
print('✅ Dossiers CSV       :',DOSSIERS_CSV)
print('✅ Champs détail      :',FIELDS_CSV)
print('✅ Anomalies          :',ANOMALIES_CSV)
print('✅ Retry requests     :',RETRY_CSV)


## 8. Étape suivante — après validation des 10 dossiers

Ne pas construire automatiquement la base finale ni recalculer `PLANNING_TL` tant que les règles de normalisation et de consolidation ne sont pas validées sur les 10 dossiers.

Une fois validé, la suite de **cette même Partie 2** ajoutera :
1. `DOMICILIATIONS` — 1 ligne par domiciliation ;
2. `DOM_VERSIONS` — initiale, augmentation 1, augmentation 2, etc. ;
3. `PLANNING_TL` — 1 ligne par mois avec `EXECUTE`, `STATUT`, `DATE_EXECUTION`, `MONTANT_EXECUTE`, `MOTIF_REJET`, `COMMENTAIRE`.

Le futur programme de migration des anciens JSON V8.x aura une seule fonction : les convertir en `DOM_EXTRACTION_V1`, puis ils passeront dans cette Partie 2 exactement comme les nouveaux dossiers.
